# Beryl worker on Kaggle

Lends Kaggle's GPUs to the Crane builder. Nothing else lives here: the site stays on Hugging Face and the code on GitHub.
This notebook only runs the worker (`scripts/beryl_gpu_server.py`): **Qwen3-ASR** listening, **Qwen3-TTS** voice and the **Qwen3.5** brain, behind an ngrok tunnel with a secret in the URL.

**Before you run it**
1. Right-hand panel, Session options: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified account).
2. Add-ons, Secrets: add `NGROK_TOKEN` (your ngrok authtoken) and attach it to this notebook. Optional: `BERYL_TOKEN` (any long random string); one is generated if you skip it.
3. Run all cells. The last cell prints the two values to paste into the builder (Admin, Voice Agent).

Cell 4 caps how long it stays up; Kaggle ends idle or long sessions on its own.

In [ ]:
# 1. What did Kaggle give us?
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print('GPUs:', torch.cuda.device_count(), '| bf16 supported:', torch.cuda.is_bf16_supported(), '(T4 has no bf16; the worker switches to fp16 on its own)')

In [ ]:
# 2. Install. Kaggle already has torch, so leave it alone.
# qwen-asr pins transformers==4.57.6 and qwen-tts pins 4.57.3; both run on 4.57.6, so qwen-tts goes in without its pin.
!pip install -q qwen-asr fastapi "uvicorn[standard]" soundfile httpx pyngrok sox einops onnxruntime
!pip install -q --no-deps qwen-tts
!apt-get install -y -qq zstd pciutils lshw sox > /dev/null
!command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Get ONLY the worker script from GitHub, plus the settings.
import os, secrets, subprocess, time, urllib.request
from kaggle_secrets import UserSecretsClient

RAW = 'https://raw.githubusercontent.com/tyronne-os/crane-builder-v1/main/scripts/beryl_gpu_server.py'
urllib.request.urlretrieve(RAW, '/kaggle/working/beryl_gpu_server.py')

sec = UserSecretsClient()
NGROK_TOKEN = sec.get_secret('NGROK_TOKEN')
try:
    BERYL_TOKEN = sec.get_secret('BERYL_TOKEN')
except Exception:
    BERYL_TOKEN = secrets.token_urlsafe(24)
BRAIN = 'hf.co/unsloth/Qwen3.5-9B-GGUF:Q4_K_M'   # ~6 GB; swap for hf.co/unsloth/Qwen3.5-4B-GGUF:Q4_K_M if it feels slow
print('worker script downloaded; secrets loaded (not printed)')

In [ ]:
# 4. Brain on GPU 1 (Ollama), ears + voice on GPU 0 (the worker).
ngpu = __import__('torch').cuda.device_count()
brain_env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1' if ngpu > 1 else '0', 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '24h'}
subprocess.Popen(['ollama', 'serve'], env=brain_env, stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(6)
print(subprocess.run(['ollama', 'pull', BRAIN], env=brain_env, capture_output=True, text=True).stdout[-300:])

worker_env = {**os.environ, 'BERYL_DEVICE': 'cuda:0', 'BERYL_TOKEN': BERYL_TOKEN, 'BERYL_VOICE_DIR': '/kaggle/working/voices',
              'HF_HOME': '/kaggle/working/hf'}
subprocess.Popen(['python', '/kaggle/working/beryl_gpu_server.py'], env=worker_env, stdout=open('/kaggle/working/beryl.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:8882/k/{BERYL_TOKEN}/health', timeout=2); break
    except Exception:
        time.sleep(2)
print('worker is up (models load on first use, so the first reply is slow)')

In [ ]:
# 5. Open the tunnel and print what to paste into the builder.
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8882, 'http')
base = f'{tunnel.public_url}/k/{BERYL_TOKEN}/v1'
print('Builder, Admin, Voice Agent:')
print('  Beryl GPU server  ->', base)
print('Builder, Admin, Workspace (GPU / model endpoint):')
print('  Model endpoint    ->', base)
print()
print('Open the builder: https://aibruh-crane-builder-v1.static.hf.space/app.html')
print('Treat that URL like a password: anyone who has it can use this GPU.')

In [ ]:
# 6. Keep the session alive and show it is healthy. Interrupt this cell to stop.
import urllib.request, json
while True:
    try:
        h = json.load(urllib.request.urlopen(f'http://127.0.0.1:8882/k/{BERYL_TOKEN}/health', timeout=5))
        print(time.strftime('%H:%M:%S'), 'ok', '| voice ready:', h['voice_ready'])
    except Exception as e:
        print(time.strftime('%H:%M:%S'), 'worker not answering:', e)
    time.sleep(300)